In [1]:
import google.auth

credentials, project_id = google.auth.default()

if hasattr(credentials, 'service_account_email'):
    print(f"Using Service Account: {credentials.service_account_email}")
elif hasattr(credentials, 'signer_email'):
    print(f"Using User Account: {credentials.signer_email}")
else:
    print("Using User Account (Identity not explicitly in object)")

print(f"Current Project ID: {project_id}")

Using User Account (Identity not explicitly in object)
Current Project ID: project-2748d8c2-1009-478f-863


In [ ]:
languages = [
    "Spanish",
    "English",
    "Korean"
]

situations = [
    "topic drifts mid-conversation",
    "user asks questions about the target language",
    "user makes a lot of errors",
    "user requests an explanation in another langauge",
    "user mixes language",
    "user expresses frustration or is stressed",
    "user gives extremely short or nonresponsive answers",
    "user tries to gaslight the model"
    "user tries to make the bot performed unwanted behavior"
]

topics = [
    "Daily Life",
    "Travel",
    "Food",
    "School",
    "Work",
    "Health",
    "Technology",
    "Culture",
    "Opinion",
    "Hobby",
    "Emotion",
    "Social Media",
    "Music",
    "Entertainment"
]

politeness = [
    "casual",
    "formal"
]


correction_styles = [
    "always correct me",
    "correct with explanation",
    "correct only major errors",
    "repeat the correct answer without pointing out",
    "never correct me"
]

correction_styles_weight = [0.1, 0.1, 0.2, 0.2, 0.4]

conversation_lengths = [
    "5-6",
    "7-8",
    "9-10",
    "11-12"
]

conversation_lengths_weight = [0.2, 0.2, 0.4, 0.2]

conversation_positions = [
    "start of the conversation",
    "in the middle, no greetings or introduction",
    "towards the end"
]

conversation_positions_weight = [0.15, 0.7, 0.15]


levels = ["A1", "A2", "B1", "B2", "C1", "C2"]

levels_weight = [0.3, 0.25, 0.2, 0.15, 0.05, 0.05]


In [10]:
sys_prompt = """
You are a dataset generator for training an AI language conversational partner.

Your task is to generate a short conversation between:
- A language learner
- A conversational partner

 Behavior Rules
- Use ONLY the target language.
- Each message should be reasonably short and conversational.
- Conversations should feel like a normal chat, not a lesson.
- Conversational partner should respond naturally. 
- Learner may make realistic mistakes that are typical for this language and CEFR level.

Output Format:
- Output valid JSON only.
- Follow the exact schema provided.
"""

In [1]:
import os
import json
import random
from google import genai
import pprint
from google.genai import types


client = genai.Client(
    vertexai=True,
    project="chatbot-dataset-483810",
    location="global"
)

for model in client.models.list():
    print(f"Model ID: {model.name}, Display Name: {model.display_name}")

Model ID: publishers/google/models/gemini-1.5-flash-002, Display Name: None
Model ID: publishers/google/models/gemini-1.5-pro-002, Display Name: None
Model ID: publishers/google/models/gemini-2.0-flash-001, Display Name: None
Model ID: publishers/google/models/gemini-2.0-flash-lite-001, Display Name: None
Model ID: publishers/google/models/gemini-2.5-flash-preview-04-17, Display Name: None
Model ID: publishers/google/models/gemini-2.5-pro-exp-03-25, Display Name: None
Model ID: publishers/google/models/gemini-2.0-flash-preview-image-generation, Display Name: None
Model ID: publishers/google/models/gemini-2.5-pro, Display Name: None
Model ID: publishers/google/models/gemini-2.5-flash, Display Name: None
Model ID: publishers/google/models/gemini-2.5-flash-lite, Display Name: None


In [12]:
N = 2000

for language in languages:

    output_path = f"{language}_training_data.jsonl"
    
    for i in range(N):
        level = random.choices(levels, weights=levels_weight)[0]
        topic = random.choice(topics)
        correction_style = random.choices(correction_styles, weights=correction_styles_weight)[0]
        conversation_length =random.choices(conversation_lengths, weights=conversation_lengths_weight)[0]
        conversation_position = random.choices(conversation_positions, weights=conversation_positions_weight)[0]
    
        usr_prompt = f"""
        Target language: {language}
        CEFR level: {level}
        Conversation topic: {topic}
        Conversation length: {conversation_length}
        Conversation position: {conversation_position}
        
        Output schema:
        {{
          "messages": [
            {{"role": "user", "content": "..."}},
            {{"role": "assistant", "content": "..."}}
          ],
          "metadata": {{
            "level": "{level}",
            "topic": "{topic}",
            "length": "{conversation_length}",
            "position": "{conversation_position}"
          }}
        }}
        """
    
        response = client.models.generate_content(
            model="gemini-3-flash-preview",
            config=types.GenerateContentConfig(
                system_instruction=sys_prompt,
                response_mime_type="application/json",
                temperature=0.9
            ),
            contents=usr_prompt
        )
    
        try:
            data_dict = json.loads(response.text)

            if i % 50 == 0:
                print(f"Generated {i} samples for {language}")
            
            with open(output_path, "a", encoding="utf-8") as f:
                f.write(json.dumps(data_dict, ensure_ascii=False) + "\n")
                
        except Exception as e:
            print(f"Error on iteration {i}: {e}")
    

Generated 0 samples for Spanish
Generated 50 samples for Spanish


KeyboardInterrupt: 